In [21]:
import requests
from lakehouse.daft import bronze, silver, gold
from lakehouse.daft.utils import daftutils
import json
import daft

In [22]:
CATALOG = "daft_catalog"

# 1. Set Up and Bronze Data

In [23]:
options = {"catalog": CATALOG, "target_schema": "bronze"}

In [24]:
@daft.udf(return_dtype=daft.DataType.string())
def get_properties(urls: daft.Series) -> list:
    result = []
    for url in urls.to_pylist():
        json_request = requests.get(url).json()
        result.append(json.dumps(json_request["result"]["properties"]))
    return result

In [25]:
class StarWarsBronze(bronze.Bronze):
    def custom_load(self, table):
        results = []
        query = f"https://swapi.tech/api/{table}"
        json_request = requests.get(query).json()
        results.extend(json_request["results"])

        while json_request["next"]:
            json_request = requests.get(json_request["next"]).json()
            results.extend(json_request["results"])
        return daft.from_pylist(results)

    def custom_transform(self, df: daft.DataFrame, table: str) -> daft.DataFrame:
        return df.with_column("properties", get_properties(daft.col("url")))
    
    def target_path(self, table: str) -> str:
        return f"D:/Data/{self.catalog}/{self.target_schema}/{table}"


bronze_instance = StarWarsBronze(**options)

In [26]:
bronze_instance.load().transform().write(mode="overwrite").execute("people")

2025-03-15 22:38:45 | people | execute | Started
2025-03-15 22:38:45 | people | load | Started
2025-03-15 22:38:50 | people | load | Completed in 0.07 min
2025-03-15 22:38:50 | people | transform | Started
2025-03-15 22:38:51 | people | transform | Completed in 0.02 min
2025-03-15 22:38:51 | people | write | Started


                                                           d

2025-03-15 22:39:25 | people | write | Completed in 0.57 min
2025-03-15 22:39:25 | people | execute | Completed in 0.67 min


In [27]:
df = daft.read_deltalake(f"D:/Data/{CATALOG}/bronze/people")
df.show()
print(f"No. Rows: {df.count_rows()}")

"LH_BronzeTSTimestamp(Microseconds, None)",nameUtf8,uidUtf8,urlUtf8,propertiesUtf8
2025-03-15 22:38:50.114682,Luke Skywalker,1,https://www.swapi.tech/api/people/1,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Luke Skywalker"", ""gender"": ""male"", ""skin_color"": ""fair"", ""hair_color"": ""blond"", ""height"": ""172"", ""eye_color"": ""blue"", ""mass"": ""77"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""19BBY"", ""url"": ""https://www.swapi.tech/api/people/1""}"
2025-03-15 22:38:50.114682,C-3PO,2,https://www.swapi.tech/api/people/2,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""C-3PO"", ""gender"": ""n/a"", ""skin_color"": ""gold"", ""hair_color"": ""n/a"", ""height"": ""167"", ""eye_color"": ""yellow"", ""mass"": ""75"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""112BBY"", ""url"": ""https://www.swapi.tech/api/people/2""}"
2025-03-15 22:38:50.114682,R2-D2,3,https://www.swapi.tech/api/people/3,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""R2-D2"", ""gender"": ""n/a"", ""skin_color"": ""white, blue"", ""hair_color"": ""n/a"", ""height"": ""96"", ""eye_color"": ""red"", ""mass"": ""32"", ""homeworld"": ""https://www.swapi.tech/api/planets/8"", ""birth_year"": ""33BBY"", ""url"": ""https://www.swapi.tech/api/people/3""}"
2025-03-15 22:38:50.114682,Darth Vader,4,https://www.swapi.tech/api/people/4,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Darth Vader"", ""gender"": ""male"", ""skin_color"": ""white"", ""hair_color"": ""none"", ""height"": ""202"", ""eye_color"": ""yellow"", ""mass"": ""136"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""41.9BBY"", ""url"": ""https://www.swapi.tech/api/people/4""}"
2025-03-15 22:38:50.114682,Leia Organa,5,https://www.swapi.tech/api/people/5,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Leia Organa"", ""gender"": ""female"", ""skin_color"": ""light"", ""hair_color"": ""brown"", ""height"": ""150"", ""eye_color"": ""brown"", ""mass"": ""49"", ""homeworld"": ""https://www.swapi.tech/api/planets/2"", ""birth_year"": ""19BBY"", ""url"": ""https://www.swapi.tech/api/people/5""}"
2025-03-15 22:38:50.114682,Owen Lars,6,https://www.swapi.tech/api/people/6,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Owen Lars"", ""gender"": ""male"", ""skin_color"": ""light"", ""hair_color"": ""brown, grey"", ""height"": ""178"", ""eye_color"": ""blue"", ""mass"": ""120"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""52BBY"", ""url"": ""https://www.swapi.tech/api/people/6""}"
2025-03-15 22:38:50.114682,Beru Whitesun lars,7,https://www.swapi.tech/api/people/7,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Beru Whitesun lars"", ""gender"": ""female"", ""skin_color"": ""light"", ""hair_color"": ""brown"", ""height"": ""165"", ""eye_color"": ""blue"", ""mass"": ""75"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""47BBY"", ""url"": ""https://www.swapi.tech/api/people/7""}"
2025-03-15 22:38:50.114682,R5-D4,8,https://www.swapi.tech/api/people/8,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""R5-D4"", ""gender"": ""n/a"", ""skin_color"": ""white, red"", ""hair_color"": ""n/a"", ""height"": ""97"", ""eye_color"": ""red"", ""mass"": ""32"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""unknown"", ""url"": ""https://www.swapi.tech/api/people/8""}"


No. Rows: 82


# 2 Silver

In [28]:
options = {
    "catalog": CATALOG,
    "source_schema": "bronze",
    "target_schema": "silver",
}

In [29]:
class StarWarsSilver(silver.Silver):

    def custom_transform(self, df: daft.DataFrame, table: str) -> daft.DataFrame:
        df = df.with_column("uid", daft.col("uid").cast(daftutils.get_daft_dtype("int")))
        for field in ["height", "mass", "gender"]:
            df = df.with_column(field, daft.col("properties").json.query(f".{field}"))
            df = df.with_column(field, daft.col(field).str.replace('"', ""))
        df = df.exclude("url", "properties")
        return df

    def source_path(self, table: str) -> str:
        return f"D:/Data/{self.catalog}/{self.source_schema}/{table}"
    
    def target_path(self, table: str) -> str:
        return f"D:/Data/{self.catalog}/{self.target_schema}/{table}"

silver_instance = StarWarsSilver(**options)

In [30]:
silver_instance.load().transform().write(mode="overwrite", overwrite_schema=True).execute(
    "people"
)

2025-03-15 22:39:25 | people | execute | Started
2025-03-15 22:39:25 | people | load | Started
2025-03-15 22:39:25 | people | load | Completed in 0.0 min
2025-03-15 22:39:25 | people | transform | Started
2025-03-15 22:39:25 | people | transform | Completed in 0.0 min
2025-03-15 22:39:25 | people | write | Started
2025-03-15 22:39:25 | people | write | Completed in 0.0 min
2025-03-15 22:39:25 | people | execute | Completed in 0.0 min


In [31]:
df = daft.read_deltalake(f"D:/Data/{CATALOG}/silver/people")
df.show()
print(f"No. Rows: {df.count_rows()}")

"LH_SilverTSTimestamp(Microseconds, None)","LH_BronzeTSTimestamp(Microseconds, None)",nameUtf8,uidInt32,heightUtf8,massUtf8,genderUtf8
2025-03-15 22:39:25.666597,2025-03-15 22:38:50.114682,Luke Skywalker,1,172,77,male
2025-03-15 22:39:25.666597,2025-03-15 22:38:50.114682,C-3PO,2,167,75,n/a
2025-03-15 22:39:25.666597,2025-03-15 22:38:50.114682,R2-D2,3,96,32,n/a
2025-03-15 22:39:25.666597,2025-03-15 22:38:50.114682,Darth Vader,4,202,136,male
2025-03-15 22:39:25.666597,2025-03-15 22:38:50.114682,Leia Organa,5,150,49,female
2025-03-15 22:39:25.666597,2025-03-15 22:38:50.114682,Owen Lars,6,178,120,male
2025-03-15 22:39:25.666597,2025-03-15 22:38:50.114682,Beru Whitesun lars,7,165,75,female
2025-03-15 22:39:25.666597,2025-03-15 22:38:50.114682,R5-D4,8,97,32,n/a


No. Rows: 82


# 3 Gold

In [32]:
options = {
    "catalog": CATALOG,
    "source_schema": "silver",
    "target_schema": "gold",
}

In [33]:
class StarWarsGold(gold.Gold):
    def people_per_gender(self, df: daft.DataFrame, table: str) -> daft.DataFrame:
        df = df.where("gender <> 'n/a'")
        df = df.where("gender <> 'none'")
        df = df.groupby("gender").agg(daft.col("name").count().alias("count"))
        return df

    def all_females(self, df: daft.DataFrame, table: str) -> daft.DataFrame:
        return df.where("gender = 'female'").exclude("LH_SilverTS", "LH_BronzeTS")
    
    def source_path(self, table: str) -> str:
        return f"D:/Data/{self.catalog}/{self.source_schema}/{table}"
    
    def target_path(self, table: str) -> str:
        return f"D:/Data/{self.catalog}/{self.target_schema}/{table}"


gold_instance = StarWarsGold(**options)

In [34]:
gold_instance.load(source_tbl="people").transform(
    tbl_transformations={
        "peoplegender": "people_per_gender",
        "peoplefemale": "all_females",
    }
).write(mode="overwrite", overwrite_schema=True).execute("peoplegender", "peoplefemale")

2025-03-15 22:39:25 | peoplegender | execute | Started
2025-03-15 22:39:25 | peoplegender | load | Started
2025-03-15 22:39:25 | peoplegender | load | Completed in 0.0 min
2025-03-15 22:39:25 | peoplegender | transform | Started
2025-03-15 22:39:25 | peoplegender | transform | Completed in 0.0 min
2025-03-15 22:39:25 | peoplegender | write | Started
2025-03-15 22:39:25 | peoplegender | write | Completed in 0.0 min
2025-03-15 22:39:25 | peoplegender | execute | Completed in 0.0 min
2025-03-15 22:39:25 | peoplefemale | execute | Started
2025-03-15 22:39:25 | peoplefemale | load | Started
2025-03-15 22:39:25 | peoplefemale | load | Completed in 0.0 min
2025-03-15 22:39:25 | peoplefemale | transform | Started
2025-03-15 22:39:25 | peoplefemale | transform | Completed in 0.0 min
2025-03-15 22:39:25 | peoplefemale | write | Started
2025-03-15 22:39:25 | peoplefemale | write | Completed in 0.0 min
2025-03-15 22:39:25 | peoplefemale | execute | Completed in 0.0 min


In [35]:
df = daft.read_deltalake(f"D:/Data/{CATALOG}/gold/peoplegender")
df.show()
print(f"No. Rows: {df.count_rows()}")

"LH_GoldTSTimestamp(Microseconds, None)",genderUtf8,countInt64
2025-03-15 22:39:25.762824,female,17
2025-03-15 22:39:25.762824,male,60
2025-03-15 22:39:25.762824,hermaphrodite,1


No. Rows: 3


In [36]:
df = daft.read_deltalake(f"D:/Data/{CATALOG}/gold/peoplefemale")
df.show()
print(f"No. Rows: {df.count_rows()}")

"LH_GoldTSTimestamp(Microseconds, None)",nameUtf8,uidInt32,heightUtf8,massUtf8,genderUtf8
2025-03-15 22:39:25.822500,Leia Organa,5,150,49,female
2025-03-15 22:39:25.822500,Beru Whitesun lars,7,165,75,female
2025-03-15 22:39:25.822500,Mon Mothma,28,150,unknown,female
2025-03-15 22:39:25.822500,Padmé Amidala,35,185,45,female
2025-03-15 22:39:25.822500,Shmi Skywalker,43,163,unknown,female
2025-03-15 22:39:25.822500,Ayla Secura,46,178,55,female
2025-03-15 22:39:25.822500,Adi Gallia,55,184,50,female
2025-03-15 22:39:25.822500,Cordé,61,157,unknown,female


No. Rows: 17


# 4 Clean Up

In [37]:
import shutil
shutil.rmtree(f"D:/Data/{CATALOG}")